In [1]:
import os
os.chdir('/opt/home/hz/power-trading/src')

%env RAY_ADDRESS=ray://prod-ray-client.zy.ai:10001

from power_trading.analysis.utils import misc
misc.logging_setup()

env: RAY_ADDRESS=ray://prod-ray-client.zy.ai:10001


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from power_trading.utils.ray import parallel_equal_pool

rto = 'pjm'

if rto == 'pjm':
    from power_trading.analysis.tools.all_positions import PjmApTools as ApTools
elif rto == 'miso':
    from power_trading.analysis.tools.all_positions import MisoApTools as ApTools
elif rto == 'spp':
    from power_trading.analysis.tools.all_positions import SppApTools as ApTools
    
aptools = ApTools()

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', 99)
pd.set_option('display.max_rows', 99)

25-04-24 16:42:14 EDT | INFO    | power_trading.utils.utils | load_env            | #471 | Load .env from /opt/home/.env...
25-04-24 16:42:14 EDT | INFO    | power_trading.config.base | load_global_config  | #496 | Loading config from local cached json file...
25-04-24 16:42:14 EDT | INFO    | power_trading.config.base | __init__            | #45  | Init global config...


In [3]:
from power_trading.data.dataset.signal.general import ConstraintsSignal, ShiftFactorSignal
from power_trading.analysis.constraints_prediction.v1.miso import MisoDayzerConstraintsPrediction as ConstraintsPrediction

In [4]:
def f(auction_month, auction_round, class_type, period_type, version):
    constraint_prediction = ConstraintsPrediction()
    market_st, market_et = aptools.tools.get_market_month_from_auction_month_and_period_trades(
        auction_month, period_type
    )

    list_cons = []
    list_cons_sum = []
    list_cons_sf = []
    list_cons_mapping = []
    total = 0
    total_sum = 0
    for market_date in pd.date_range(market_st, market_et, freq='MS', inclusive='left'):
        auction_month_str = auction_month.strftime('%Y-%m')
        market_date_str = market_date.strftime('%Y-%m')

        p = Path(
            f'/opt/data/tmp/pw_data/spice/pw_flows/{rto}_annual_perturb/auction_month={auction_month_str}/market_month={market_date_str}/auction_round={auction_round}/'
        )
        
        limit = pd.read_parquet(p / 'limit.parquet').squeeze()
        useful_limit = limit[limit >= 10]
        percentage = pd.read_parquet(p / 'flow_memo.parquet')
        percentage = percentage.loc[:, percentage.columns.get_level_values('constraint_id').isin(useful_limit.index)]
        # percentage[percentage > 10] = 0
        mask = ~(percentage > 1.5).all(axis=0)
        percentage = percentage.loc[:, mask]
        # hour_list = [12, 18] if class_type == 'onpeak' else [0, 5]
        hour_list = [0, 5, 12, 18]
        percentage = percentage[percentage.index.get_level_values('se_date').hour.isin(hour_list)]
        percentage1 = pd.read_parquet(p / f'flow_onpeak.parquet') if class_type == 'onpeak' else pd.read_parquet(p / f'flow_offpeak.parquet')
        percentage1 = percentage1.loc[:, percentage1.columns.get_level_values('constraint_id').isin(useful_limit.index)]
        # percentage1[percentage1 > 10] = 0
        mask = ~(percentage1 > 1.5).all(axis=0)
        percentage1 = percentage1.loc[:, mask]
        # thres = 0.8
        # mask = percentage.max() >= thres
        # mask1 = percentage1.max() >= thres
        # common_cols = percentage.columns[mask].intersection(percentage1.columns[mask1])
        # percentage = percentage[common_cols]
        # percentage1 = percentage1[common_cols]
        # percentage = pd.concat([percentage, percentage1], axis=0).fillna(0)
        # percentage = percentage1
        tmp, tmp_sum = [], []
        max_lvl = 105
        total_cols = list(range(60, max_lvl, 5))
        for col in total_cols:
            gp = (percentage.round(2) >= col / 100).groupby(level='outage_date')
            constraints = gp.max().sum()
            constraints = constraints[constraints > 0].rename(str(col))
            tmp.append(constraints)
            
            constraints = gp.sum().sum()
            constraints = constraints[constraints > 0].rename(str(col))
            tmp_sum.append(constraints)
        df_tmp = pd.concat(tmp, axis=1)
        df_tmp_sum = pd.concat(tmp_sum, axis=1)
        list_cons.append(df_tmp.copy())
        list_cons_sum.append(df_tmp_sum.copy())
        df_tmp /= percentage.index.get_level_values('outage_date').nunique()
        df_tmp.insert(0, '0', 1 - df_tmp.sum(axis=1))
        weight = np.array([7**i for i in range(len(total_cols) + 1)])
        ttmp = constraint_prediction.constraints_flow_deviation_distance(df_tmp, weight=weight)
        ttmp.index = ttmp.index.droplevel('flow_direction')
        list_cons_sf.append(ttmp.groupby(level=0).max().rename(market_date))
        total_sum += percentage.shape[0]
        total += percentage.index.get_level_values('outage_date').nunique()
        cons_mapping = pd.read_parquet(
            f'/opt/data/tmp/pw_data/spice/prod_annual_model_{rto}/auction_month={auction_month_str}/market_month={market_date_str}/auction_round={auction_round}/res/{market_date_str}-01/constraints.parquet'
        )
        list_cons_mapping.append(cons_mapping)
    cons_mapping = pd.concat(list_cons_mapping, axis=0)
    cons = pd.concat(list_cons, axis=0).fillna(0)
    cons_sum = pd.concat(list_cons_sum, axis=0).fillna(0)
    tmp_df = pd.concat(list_cons_sf, axis=1).T.groupby(level=0).max().idxmax()
    list_sf = []
    for market_month in tmp_df.unique():
        print(market_month)
        market_date_str = market_month.strftime('%Y-%m')
        p = Path(f'/opt/data/tmp/pw_data/spice/prod_annual_model_{rto}/auction_month={auction_month_str}/market_month={market_date_str}/auction_round={auction_round}/res')
        st = market_month
        et = st + pd.offsets.MonthBegin(1)
        inner_sf = None
        count = 0
        idx = tmp_df[tmp_df == market_month].index
        for sf_date in pd.date_range(st, et, freq='3D', inclusive='left'):
            tmp_sf = pd.read_parquet(p / sf_date.strftime('%Y-%m-%d') / 'sf.parquet')
            tmp_sf = tmp_sf[tmp_sf.columns.intersection(idx)]
            count += 1
            if inner_sf is None:
                inner_sf = tmp_sf
            else:
                inner_sf = inner_sf.add(tmp_sf, fill_value=0)
        inner_sf /= count
        list_sf.append(inner_sf)
    sf = pd.concat(list_sf, axis=1)
    # sf1 = pd.read_parquet(f'/opt/data/tmp/tmp/sf/pjm_sf_test_agg_constraint_id_lam20_daily30/{auction_month_str}.parquet')
    # common_constraints = sf.columns.intersection(sf1.columns)
    # sf1 = sf1[common_constraints].copy()
    # sf = pd.concat([sf1, sf[sf.columns.difference(sf1.columns)]], axis=1)
    print(f'sf shape is {sf.shape}')
    signal_ori1 = cons.groupby(level=['flow_direction', 'constraint_id']).sum()
    signal_ori_sum1 = cons_sum.groupby(level=['flow_direction', 'constraint_id']).sum()
    signal_ori = signal_ori1[signal_ori1.columns[::-1]].fillna(0).diff(axis=1)[signal_ori1.columns]
    signal_ori_sum = signal_ori_sum1[signal_ori_sum1.columns[::-1]].fillna(0).diff(axis=1)[signal_ori_sum1.columns]
    signal_ori[f"{max_lvl-5}"] = signal_ori[f"{max_lvl-5}"].fillna(signal_ori1[f"{max_lvl-5}"])
    signal_ori_sum[f"{max_lvl-5}"] = signal_ori_sum[f"{max_lvl-5}"].fillna(signal_ori_sum1[f"{max_lvl-5}"])
    signal_ori /= total
    signal_ori_sum /= total_sum
    signal_ori.insert(0, "0", 1 - signal_ori.sum(axis=1))
    signal_ori_sum.insert(0, "0", 1 - signal_ori_sum.sum(axis=1))
    weight = np.array([7**i for i in range(len(total_cols) + 1)])
    deviation = np.log1p(constraint_prediction.constraints_flow_deviation_distance(signal_ori, weight=weight))
    deviation_sum = np.log1p(constraint_prediction.constraints_flow_deviation_distance(signal_ori_sum, weight=weight))
    signal_ori = pd.concat([signal_ori.add_suffix('_max'), signal_ori_sum.add_suffix('_sum')], axis=1)
    signal_ori = signal_ori.join(deviation.rename('deviation_max')).join(deviation_sum.rename('deviation_sum'))
    # signal_ori['deviation'] = signal_ori[['deviation_max', 'deviation_sum']].mean(axis=1)
    # signal_ori['deviation'] = signal_ori[['deviation_sum']].mean(axis=1)
    signal_ori['flow_direction'] = signal_ori.index.get_level_values('flow_direction')
    signal_ori.index = signal_ori.index.droplevel('flow_direction')

    mapping = cons_mapping[cons_mapping['convention'] < 10].drop_duplicates(
        subset=['constraint_id']
    ).set_index('constraint_id')[['branch_name', 'convention']].rename(
        columns={'branch_name': 'equipment'}
    )
    signal_ori = signal_ori.join(mapping, how='left').reset_index()
    signal_ori['shadow_sign'] = -signal_ori['flow_direction']

    if auction_round == 1:
        run_at = auction_month.replace(month=4, day=1)
    elif auction_round == 2:
        run_at = auction_month.replace(month=4, day=17)
    elif auction_round == 3:
        run_at = auction_month.replace(month=4, day=24)
    elif auction_round == 4:
        run_at = auction_month.replace(month=4, day=22)
    et = run_at + pd.DateOffset(days=1)
    st = et - pd.DateOffset(years=2)
    if st.month < 6:
        st -= pd.DateOffset(years=1)
    st = st.replace(month=6)
    da = aptools.tools.get_da_shadow_by_peaktype(st=st, et_ex=et, peak_type=class_type, offpeak_hrs=(22, 9))
    da['shadow_sign'] = np.where(da['shadow_price'] > 0, 1, -1)
    da['constraint_id'] = da['monitored_facility'] + ':' + da['contingency_facility'].replace('ACTUAL', 'BASE')
    da = da.merge(mapping.reset_index(), on='constraint_id', how='left')
    da_sp = da.groupby(['equipment', 'convention', 'shadow_sign'])['shadow_price'].sum().abs().reset_index()
    da_sp['convention'] = da_sp['convention'].astype(int)
    signal_ori = signal_ori.merge(da_sp, on=['equipment', 'convention', 'shadow_sign'], how='left')
    signal_ori['shadow_price_da'] = signal_ori['shadow_price'].fillna(0)

    signal = signal_ori.set_index('constraint_id')
    signal['constraint'] = signal.index.str.split(':').str[0]
    cols = [str(x) + y for x in range(90, max_lvl, 5) for y in ['_max', '_sum']]
    signal = signal[(signal[cols] > 0).any(axis=1)].copy()

    signal['deviation_max_rank'] = signal['deviation_max'].rank(ascending=False, pct=True)
    signal['deviation_sum_rank'] = signal['deviation_sum'].rank(ascending=False, pct=True)
    signal['shadow_rank'] = signal['shadow_price_da'].rank(ascending=False, pct=True)
    signal['rank'] = signal['deviation_max_rank'] * 0.3 + signal['deviation_sum_rank'] * 0.2 + signal['shadow_rank'] * 0.5
    
    _signal = signal.sort_values(['shadow_rank', 'rank'], ascending=[True, True]).drop_duplicates(
        # subset=['equipment', 'convention'], keep='first'
        subset=['equipment'], keep='first'
    )
    signal_rank = signal.groupby(['equipment'])['rank'].min().reset_index()
    signal = _signal.reset_index().drop(columns=['rank']).merge(signal_rank, on=['equipment'], how='left')
    # signal = _signal.copy()
    signal['equipment'] = signal['equipment'] + ',' + signal['constraint']
    common_cons = sf.columns.intersection(signal['constraint_id'])
    signal = signal[signal['constraint_id'].isin(common_cons)].set_index('constraint_id')
    sf = sf[signal.index].copy()
    
    signal['rank'] = signal['rank'].rank(ascending=True, pct=True)
    signal['shadow_price'] = -signal['flow_direction']
    signal.index += '|' + (-signal['flow_direction']).astype(str) + '|spice'
    sf.columns = signal.index
    signal['tier'] = 5
    if class_type == 'onpeak':
        # v6
        signal.loc[signal['rank'] <= 1.00*1, 'tier'] = 4
        signal.loc[signal['rank'] <= 0.72*1, 'tier'] = 3 # 0.24
        signal.loc[signal['rank'] <= 0.48*1, 'tier'] = 2 # 0.20
        signal.loc[signal['rank'] <= 0.28*1, 'tier'] = 1 # 0.16
        signal.loc[signal['rank'] <= 0.12*1, 'tier'] = 0 # 0.12
        # v5
        # signal.loc[signal['rank'] <= 1.0*1, 'tier'] = 4
        # signal.loc[signal['rank'] <= 0.8*1, 'tier'] = 3
        # signal.loc[signal['rank'] <= 0.6*1, 'tier'] = 2
        # signal.loc[signal['rank'] <= 0.4*1, 'tier'] = 1
        # signal.loc[signal['rank'] <= 0.2*1, 'tier'] = 0
    else:
        # signal.loc[signal['rank'] <= 1.0*1, 'tier'] = 4
        # signal.loc[signal['rank'] <= 0.8*1, 'tier'] = 3
        # signal.loc[signal['rank'] <= 0.6*1, 'tier'] = 2
        # signal.loc[signal['rank'] <= 0.4*1, 'tier'] = 1
        # signal.loc[signal['rank'] <= 0.2*1, 'tier'] = 0
        signal.loc[signal['rank'] <= 1.00*1, 'tier'] = 4
        signal.loc[signal['rank'] <= 0.72*1, 'tier'] = 3 # 0.24
        signal.loc[signal['rank'] <= 0.48*1, 'tier'] = 2 # 0.20
        signal.loc[signal['rank'] <= 0.28*1, 'tier'] = 1 # 0.16
        signal.loc[signal['rank'] <= 0.12*1, 'tier'] = 0 # 0.12
    signal = signal[signal['tier'] <= 4].copy()
    print(signal.groupby('tier').size())
    signal_name = f'TEST.Signal.{rto.upper()}.SPICE_ANNUAL_V4.{version}.R{auction_round}'
    
    loader = ConstraintsSignal(aptools.rto, signal_name, period_type, class_type)
    sf_loader = ShiftFactorSignal(aptools.rto, signal_name, period_type, class_type)
    loader.save_data(data=signal, auction_month=auction_month)
    sf_loader.save_data(data=sf, auction_month=auction_month)
    
    if class_type == 'onpeak':
        loader = ConstraintsSignal(aptools.rto, signal_name, period_type, 'wkndonpeak')
        sf_loader = ShiftFactorSignal(aptools.rto, signal_name, period_type, 'wkndonpeak')
        loader.save_data(data=signal, auction_month=auction_month)
        sf_loader.save_data(data=sf, auction_month=auction_month)

In [5]:
params = [
    {
        'auction_month': pd.Timestamp(year=year, month=6, day=1),
        'auction_round': auction_round,
        'class_type': class_type,
        'period_type': period_type,
        'version': v,
    }
    for year in [2025]
    for auction_round in [3]
    for period_type in ['a']
    for class_type in ['onpeak', 'dailyoffpeak']
    # for class_type in ['onpeak']
    for v in [6]
]

len(params)

2

In [6]:
# f(**params[0])
parallel_equal_pool(f, params, n_jobs=24)

25-04-24 16:42:32 EDT | INFO    | power_trading.utils.ray | parallel_equal_pool | #364 | RAY_ADDRESS is set to 'ray://prod-ray-client.zy.ai:10001', using ray (ignore 'no_ray (False)')...
25-04-24 16:42:32 EDT | INFO    | power_trading.utils.ray | parallel_equal_pool | #395 | Do 2 tasks parallelly in ray 2 processes...
25-04-24 16:42:32 EDT | INFO    | /opt/home/hz/power-trading/src/power_trading/config/config.py | init_ray            | #162 | Found requirements.txt, installing /opt/home/hz/power-trading/requirements-ray.txt ...
2025-04-24 16:42:32,711	INFO client_builder.py:244 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.
25-04-24 16:42:35 EDT | INFO    | /opt/home/hz/power-trading/src/power_trading/config/config.py | init_ray            | #199 | Remote global config (ClientActorHandle(79923463f006f61eb30c2ef101000000))loaded...


  0%|          | 0/2 [00:00<?, ?it/s]

(PoolActor pid=2399140, ip=10.42.98.95) 2025-06-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2025-07-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2025-08-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2025-06-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2025-09-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2025-07-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2026-02-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2025-08-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2026-01-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2025-09-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2026-05-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2025-10-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2026-02-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2026-01-01 00:00:00
(PoolActor pid=2399140, ip=10.42.98.95) 2026-03-01 00:00:00
(PoolActor pid=2348811, ip=10.42.141.174) 2026-05-01 00:00:00
(PoolActor pid=2399140, ip

(PoolActor pid=2399140, ip=10.42.98.95) 25-04-24 13:44:53 PDT | INFO    | power_trading.analysis.tools.base | add_offpeak         | #6787 | extended offpeak: 29008 hours => 45196 hours


(PoolActor pid=2399140, ip=10.42.98.95) tier
(PoolActor pid=2399140, ip=10.42.98.95) 0    132
(PoolActor pid=2399140, ip=10.42.98.95) 1    177
(PoolActor pid=2399140, ip=10.42.98.95) 2    220
(PoolActor pid=2399140, ip=10.42.98.95) 3    265
(PoolActor pid=2399140, ip=10.42.98.95) 4    310
(PoolActor pid=2399140, ip=10.42.98.95) dtype: int64
(PoolActor pid=2399140, ip=10.42.98.95) <class 'pandas.core.frame.DataFrame'>
(PoolActor pid=2399140, ip=10.42.98.95) Index: 1104 entries, LENOX-NMESHOPP NML 1090     B  115 KV:TEMP:DBL.L230.Lack-Oxbow+L230.ETowanda-Hillside|-1|spice to 16 WAUKE138 KV  TR81CT-S:138L1604 Waukegan-1604-Libertyville 138 kV Line|-1|spice
(PoolActor pid=2399140, ip=10.42.98.95) Data columns (total 34 columns):
(PoolActor pid=2399140, ip=10.42.98.95)  #   Column              Non-Null Count  Dtype  
(PoolActor pid=2399140, ip=10.42.98.95) ---  ------              --------------  -----  
(PoolActor pid=2399140, ip=10.42.98.95)  0   0_max               1104 non-null   float6

(PoolActor pid=2348811, ip=10.42.141.174) 25-04-24 13:45:16 PDT | INFO    | power_trading.analysis.tools.base | add_offpeak         | #6787 | extended offpeak: 29008 hours => 45196 hours


(PoolActor pid=2348811, ip=10.42.141.174) tier
(PoolActor pid=2348811, ip=10.42.141.174) 0    132
(PoolActor pid=2348811, ip=10.42.141.174) 1    177
(PoolActor pid=2348811, ip=10.42.141.174) 2    220
(PoolActor pid=2348811, ip=10.42.141.174) 3    265
(PoolActor pid=2348811, ip=10.42.141.174) 4    310
(PoolActor pid=2348811, ip=10.42.141.174) dtype: int64
(PoolActor pid=2348811, ip=10.42.141.174) <class 'pandas.core.frame.DataFrame'>
(PoolActor pid=2348811, ip=10.42.141.174) Index: 1104 entries, LENOX-NMESHOPP NML 1090     B  115 KV:TEMP:DBL.L230.Lack-Oxbow+L230.ETowanda-Hillside|-1|spice to 16 WAUKE138 KV  TR81CT-S:138L1604 Waukegan-1604-Libertyville 138 kV Line|-1|spice
(PoolActor pid=2348811, ip=10.42.141.174) Data columns (total 34 columns):
(PoolActor pid=2348811, ip=10.42.141.174)  #   Column              Non-Null Count  Dtype  
(PoolActor pid=2348811, ip=10.42.141.174) ---  ------              --------------  -----  
(PoolActor pid=2348811, ip=10.42.141.174)  0   0_max           

[None, None]

In [7]:
sf = pd.read_parquet('/opt/data/xyz-dataset/signal_data/pjm/sf/TEST.Signal.PJM.SPICE_ANNUAL_V4.4.R1/2024-06/a/onpeak/')

In [19]:
sf.loc['1047909']

,NOTTINGHM 2-3 SER DEV A 230 KV:DBL.L500.Conastone-Hunt.5013 & Conastone-P.B.5012|-1|spice,CONASTON-NORTHWES 2322 B 230 KV:DBL:L500.Brghtn-Conastn & L230.Constne-NrthWst2310|-1|spice,Pres-Tibb l/o St-Francis-Lutesville 345kV:St. Francis-Lutesville 345kV|-1|spice,GRACETON-SAFEHARB 2303 A 230 KV:DBL:Cona-PB 5012 + Cooper-PB Tap-Nttnghm 220-08|-1|spice,LENOX-NMESHOPP NML 1090 B 115 KV:TEMP:DBL.L230.Lack-Oxbow+L230.ETowanda-Hillside|-1|spice,BOONETWN-SREADING BOO-SRE A 230 KV:TEMP:500/230.Lauschtown T3 + Conastone-PB 5012|-1|spice,GARDNERS-TEXSEAST GAR-TEX A 115 KV:DBL.L500.Hunter-Conast.5013 & L230.Hunter-Jack|-1|spice,HOWARD2 138 KV HOW-MEL1:L345.SouthBerwick-Galion|-1|spice,Chicago - Praxair3 138 kV l/o Wilton Center - Dumont 765 kV:Dumont-Wilton Center 765kV|-1|spice,Sandburg XFMR 3 161/138 kV l/o Oak Grove - Sandburg 345 kV:Oak Grove - Sandburg 345 kV|-1|spice,ROXBURY 115 KV ROX-SHA:L500.Conastone-PeachBottom.5012|-1|spice,JUNIATA 2 XFORMER H 500 KV:TEMP: Juniata T1 & TMI T1|-1|spice,DESOTO-SELAMPAR DES-SEL A 138 KV:L138.Modoc-Desoto|-1|spice,CUMB PL-JUNIATA 2235 B 230 KV:Juniata - TMI 5008 & Juniata - Dauphin|-1|spice,ALLDAM6-KITTANNI SP-KT A 138 KV:L138.KiskiValley-AlleghenyLudlum2-ShaffersCorner|-1|spice,PLEASNTV-ASHBURN 274D A 230 KV:L500.CalvertCliffs-ChalkPoint.5072|-1|spice,Crescent Ridge - 7713 138 l/o Kewanee East - Bureau:L138.KewaneeEast-Bureau (Ameren)|-1|spice,ELIMA 138 KV ELI-HAV1:L345.EastLima-MaddoxCreek|-1|spice,FREMNT - WFREMONT TIE A 138 KV:BASE|-1|spice,FACEROCK69 KV FAROZBR:L500.Conastone-PeachBottom.5012|-1|spice,12 DRESDEN 45TR81 CT H 345 KV:L345.12 Dresden-900 Elwood.1220|-1|spice,COOLSPRI-MILFORD 23069 A 230 KV:L230.IndianRiver-Milford.23034 & L230.Steele-Vienna.23085 & 230/138.Vienna.AT20|-1|spice,ETOWANDA-HILLSIDE 2002 TIE A 230 KV:TEMP:Lackawana-Oxbow + Lobo-Moshannon|-1|spice,HIGHLAOE-COMERC A 138 KV:DBL:Hanna-Highland 345kv + Sammis-Wellsville 345kv|-1|spice,HOPECREE-SILVERUN TIE B 230 KV:TEMP: 5015/5025 500kv DBL|-1|spice,IDYLWOO4-CLARK4 202A A 230 KV:L230.Beaumead-Ashburn-PleasantView.274|-1|spice,MapleSt - Chrysler 138 kV l/o HighlandPark - NewLondon 230 kV:L230.NewLondon-KokomoHighlandPark|-1|spice,Powerton-Towerline 138 kV l/o Fargo-Sandburg 345 kV:L345.Fargo-Sandbrg|-1|spice,CONASTON 500-4 XFORMER H 500 KV:TEMP:DBL.Conastone.500-2+Brighton.Conastone.5011|-1|spice,CONASTON-PEACHBOT 5012 B 500 KV:BASE|-1|spice,LINCOLN 115 KV LIN-STR:L500.Brighton-Conastone.5011|-1|spice,ALLENIM-RPMONE ALL-RPM A 345 KV:L765.HangingRock-Jefferson|-1|spice,CEDARCRE230 KV CED-SIL:L230.Cartanza-Milford.23033|-1|spice,Shadland-Lafayette_South 138 kV l/o Westwood-W_Lafayette 138 kV:Westwood-08NW_Tap-WLafyett 138+Purdue-Lafaycin 138|-1|spice,SCOTTSVI-BREMO_VP A 138 KV:L500.Cloverdale-Lexington.566 & Cloverdale 6A/6B|-1|spice,CONASTON 500-2 XFORMER H 500 KV:L500.Brighton-Conastone.5011|-1|spice,BEAUMEAD230 KV 274C:L230.Brambleton-Beaumeade.227|-1|spice,CEDARSUB-WILLIAM F-2206-1 A 230 KV:5018 and Cedargrove-Roseland Y-2277|-1|spice,SAYRECON230 KV SAY-SAY:BASE|-1|spice,College Corner - Collinsville 138 kV l/o Tanners Creek - Miami Fort 345 kV:L345.TannersCreek-MiamiFort|-1|spice,New London-Kokomo 230kV l/o Cayuga-Nucor 345kV:L345.Cayuga-Nucor (CIN)|-1|spice,EBERSOLE138 KV EBE-FOS2:L138.Ebersole-FostoriaCentral.1|-1|spice,Lafayette 138/230 kV TR1 l/o Lafayette 138/230 TR4:Lafayette 138/230 kV TR4|-1|spice,COOPERPE230 KV COO-PEA:L500.Conastone-PeachBottom.5012|-1|spice,FARMVILL230 KV TX5:L230.Clover-Briery-Farmville.235|-1|spice,Turkey Hill - Hilgard 138 kV l/o Prairie State - Mt Vernon 345 kV:Prairie State - Mt Vernon 345 kV|-1|spice,BRUNSWIC230 KV BUR-MEA:L230.Deans-MeadowRoad-PiersonAve-Metchuen.S-2219|-1|spice,HARWOOD-SUSQUEHA B 230 KV:Harwood and Wescosville T3|-1|spice,LENNOX-MACNEW_T LEN-MAC A 115 KV:L230.ETowanda-Hillside.2002 + Relay.ESayre|-1|spice,...,Kincaid - Austin 345 kV l/o Pana - Kincaid 345 kV:Pana-Kincaid 345kV|-1|spice,LAKE_REB-UNION_EK TIE B 138 KV:L138.Fawkes-FawkesLG (Tie)|-1|spice,ROCKWOOD115 KV RO

In [13]:
from power_trading.data.dataset.ftr.network.miso import MisoMonitoredLineTransformer

MisoMonitoredLineTransformer().load_data(auction_month=pd.Timestamp('2025-04'), period_type='f0')

,device_name,device_type,base_case_rating,emergency_rating,class_type,market_round,market_period,market_name
0,MAGIC_CY TR7,XFMR,302.40,347.76,onpeak,1,f0,apr_2025
1,STILLWEL NO_4_XFMR,XFMR,60.30,60.30,onpeak,1,f0,apr_2025
2,EL_EHV AT2,XFMR,805.50,805.50,onpeak,1,f0,apr_2025
3,ARGENTA TB1,XFMR,473.40,506.70,onpeak,1,f0,apr_2025
4,STL500 AT1,XFMR,538.20,538.20,onpeak,1,f0,apr_2025
...,...,...,...,...,...,...,...,...
25471,ADMN_ADMF_1651 A,LINE,285.60,365.50,offpeak,1,f1,apr_2025
25472,ELLISELSSJ13_1 1,LINE,135.15,178.50,offpeak,1,f1,apr_2025
25473,DONAOLIV69_1 1,LINE,49.30,49.30,offpeak,1,f1,apr_2025
25474,BNNW_JORD_1517 A,LINE,359.55,406.30,offpeak,1,f1,apr_2025
